In [2]:
from debugbox import DOSBoxDebugger
from pathlib import Path
import time
from collections import Counter
import subprocess

In [3]:
appname='/opt/homebrew/bin/dosbox-x'

def activate(app):
    subprocess.run(["osascript", "-e", f'tell application "System Events" to set frontmost of process "{app}" to true'])
def send_key_code(code, app=appname, modifier=None):
    subprocess.run(["osascript", "-e", f'tell application "{appname}" to activate'])
    mod = f" using {{{modifier}}}" if modifier else ""
    script = f'''
      tell application "System Events"
      tell process "{app}"
          key code {code}{mod}
      end tell
      end tell'''
    subprocess.run(["osascript", "-e", script])

In [20]:
import re
ANSI_RE = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')

def parse_mcb(raw):
    
    clean = ANSI_RE.sub('', raw)
    mcb = {}
    
    in_table = False
    
    for line in clean.splitlines():
        line = line.strip()
    
        if line.startswith("LOG: MCB Seg"):
            in_table = True
            continue
    
        if not in_table:
            continue
    
        # Stop when we hit the next section
        if line == "LOG: Upper memory:":
            break
    
        # Remove the LOG: prefix
        if line.startswith("LOG:"):
            line = line[4:].strip()
    
        # Ignore blank lines and END OF CHAIN
        if not line or "END OF CHAIN" in line:
            continue
    
        parts = line.split()
    
        # Need at least:
        # segment, size, PSP
        if len(parts) < 3:
            continue
    
        segment = parts[0]
        size = int(parts[1])
        psp = parts[2]
    
        # Remaining text is notes / filename
        rest = parts[3:]
    
        notes = None
        filename = None
    
        if rest:
            # "(DOS)" / "(free)" etc.
            if rest[0].startswith("("):
                notes = rest[0]
                rest = rest[1:]
    
            if rest:
                filename = rest[0]
    
        mcb[segment] = {
            "size": size,
            "psp": psp,
            "notes": notes,
            "filename": filename,
        }
    return mcb

def find_owner(hex_addr, mcb):
    addr = int(hex_addr, 16)

    for segment, block in mcb.items():
        start = int(segment, 16)
        end = start + (block["size"] // 16)

        if start <= addr < end:
            return {
                "segment": segment,
                "filename": block["filename"],
                "size": block["size"],
                "start": start,
                "end": end,
            }

    return None

In [16]:
def get_logcpu_file():
    p = Path('/Users/robmaxwell/projects/f19/LOGCPU.TXT')
    text = p.read_text(errors="replace")
    return text

def get_pairs(text):
    pairs = [
        line.strip()
        for line in text.splitlines()
        if ":" in line
    ]
    counts = Counter(pairs)
    return counts

def get_transitions(text):
    pairs = [
        line.strip()
        for line in text.splitlines()
        if ":" in line
    ]
    transitions = Counter(
        zip(pairs[:-1], pairs[1:])
    )
    return transitions

In [7]:
def start_to_roster(bp):
    dbg.set_breakpoint(bp)    
    dbg.run()
    send_key_code(45, app=appname)
    time.sleep(1)
    send_key_code(18, app=appname)
    for i in range(5):
        send_key_code(36, app=appname)
        time.sleep(1)

In [29]:
dbg = DOSBoxDebugger(
    conf_path="/Users/robmaxwell/Library/Preferences/f19-dosbox.conf",
    dump_path="/Users/robmaxwell/projects/f19/MEMDUMP.BIN",
)
dbg.start()

AUTOTYPE ENTER
LOG: Early LOG Init complete
LOG: CONFIG: Loaded config file: /Users/robmaxwell/Library/Preferences/f19-dosbox.conf
LOG: DOSBox-X's working directory: /Users/robmaxwell/projects/f19
LOG: Logging init: beginning logging proper. This is the end of the early init logging
LOG: Logging: opened logfile '/Users/robmaxwell/projects/f19/logs/f19-dosbox.log' successfully. All further logging will go to this file.
LOG: DOSBox-X version 2026.08.02 Commit e522642 (macOS ARM SDL2 64-bit)
LOG: macOS EXE path: /opt/homebrew/bin/dosbox-x
LOG: macOS Resource path: 
LOG: macOS: NSTouchBar support detected in system
LOG: Prevent capture: 0
LOG: SDL: version 2.32.70, Video cocoa, Audio coreaudio
LOG: Host keyboard layout is now us (US English)
LOG: Mapper keyboard layout is now us (US English)
LOG: SDL2 reports desktop display mode 1470 x 956
LOG: The default output for the video system: opengl
LOG: OpenGL Version : 2.1 Metal - 89.4
LOG: Configured windowposition: -
LOG: SDL: Current window 

In [30]:
start_to_roster('1CE8:00004877')


B_
P_
 _
1_C_E_8_:_0_0_0_0_4_8_7_7_LOG: DEBUG: Set breakpoint at 1CE8:4877

_                
R_
U_
N_
(Running)                                                                      

0814:00000100 BC2607              mov  sp,0726                                  0814:00000103 BB7300              mov  bx,0073                                  00000103000007261
666667 LOG: pixratio 1.200, dw true, dh false
LOG: Aspect ratio: 640 x 480  xToY=1.333 yToX=0.750
CA DB E8 1C.00 47 E8 1CG
..

1CE8:00004877 C7062A150002        mov  word [152A],0200       ds:[152A]=0000    1CE8:0000487D 5D                  pop  bp                                       1CE8:0000487E 07  pop  es   
1CE8:0000487F 5E  pop  si
1CE8:00004880 5Fop  di
1CE8:00004881 8BE5sp,bp
I-> _                                                                          502CE80002CE82CE87D34  CS=1CE8000048771110DAB804F0000DAB04503816
4147.958

In [31]:
d = {}
dbg.run()
send_key_code(45, app=appname)
time.sleep(1)
send_key_code(18, app=appname)
for i in range(5):
    send_key_code(36, app=appname)
dbg.send("BPDEL *")

for i in range(45):
    dbg.send("LOGC 100000")
    dbg.run()
    send_key_code(36, app=appname)
    dbg.wait_for_log_complete()
    text = get_logcpu_file()
    pairs = get_pairs(text)
    transitions = get_transitions(text)
    dbg.send("DOS MCBS")
    mcb_log.seek(last_pos)
    new_text = mcb_log.read()
    last_pos = mcb_log.tell()
    mcbs = parse_mcb(new_text)
    p = Path('/Users/robmaxwell/projects/f19/LOGCPU.TXT')
    p.rename(f'logcpu_{i}.txt')
    d[i] = (pairs, transitions, mcbs)

mcb_log.close()


R_
U_
N_
(Running)                                                                      

1CE8:00004877 C7062A150002        mov  word [152A],0200       ds:[152A]=0200    1CE8:0000487D 5D                  pop  bp                                       0000487D7
59333333 

1CE8:00004877 C7062A150002        mov  word [152A],0200       ds:[152A]=0022    1CE8:0000487D 5D                  pop  bp                                       
I-> _                                                                          F
000048770
5B6090872
818.596666667 
B_
P_
D_
E_L_ _*_LOG: DEBUG: Breakpoints deleted.

_       
L_
O_
G_
C_ _1_0_0_0_0_0_LOG: DEBUG: Starting log

(Running)                                                                      LOG: DEBUG: cpu log LOGCPU.TXT created


F000:0000D10B E2FE        loop 0000D10B ($-2)                
F000:0000D10D 59cx
F000:0000D10E CFiret   
F000:0000D10F 0000add  [bx+si],alds:[1005]=0000
F000:0000D111 0000add  [bx+si],alds:[1005]=0000
F000:0000D113 0000a

TimeoutError: Timed out waiting for DOSBox-X to finish LOGC

LOG: DEBUG: cpu log LOGCPU.TXT created
9C 19 23 0A..#8C 10 23 0A#
.

F000:0000D184 FB    sti         
F000:0000D185 F4    hlt         
F000:0000D186 CF  iret      
F000:0000D187 0000add  [bx+si],alds:[000F]=6954
F000:0000D189 0000add  [bx+si],alds:[000F]=6954
F000:0000D18B 0000add  [bx+si],alds:[000F]=6954   
I-> _                                                                          110D001D1F88D1FE201F000D18410011FFE16201F781048575
2165763.504333333 

In [62]:
#dbg.run()
#zz[0]['pairs']
#{k: [type(dict(v[0])), type(dict(v[1])), type(v[2])] for k, v in d.items()}
#[len(i) for i in list(zz[0]['transitions'].keys())]

In [67]:
#list(d.keys())[0]
#{str(key): value for key, value in dict(v[1]).items()}

def ugh(v):
    return {str(key): value for key, value in dict(v).items()}
    
zz = {k: {'pairs': dict(v[0]), 'transitions': ugh(v[1]), 'mbcs': v[2]} for k, v in d.items()}

In [68]:
import json
with open("log_cpu_data.json", "w") as file:
    json.dump(zz, file)

>K8